# Module 307 — XSS and Safe LLM Output Handling

This worksheet accompanies HTB Section 2. It demonstrates the difference between text that a browser could interpret as markup and the same text encoded for display. It uses only harmless local strings.

The notebook does not render executable HTML, run JavaScript, host a web server, access cookies, send requests, or connect to an HTB target.

## The browser interprets context

The string `'<b>Hello</b>'` can be interpreted as HTML when inserted into an HTML page. If the application intended to display the characters literally, that is an output-handling mistake. The safe treatment is context-specific: HTML-escape the value, or use an API that inserts it as text (such as a browser DOM element's `textContent`).

The code below prints both source strings as ordinary notebook output. It deliberately does **not** pass the unsafe one to an HTML renderer.

In [ ]:
from html import escape

# Treat this as untrusted text returned by a model or submitted by a user.
untrusted_text = "<b>Hello</b> & welcome"

# If a web app inserts this value as HTML, the browser may parse <b> as a tag.
# We only print the source here; we do not render it as HTML.
unsafe_html_source = untrusted_text

# html.escape replaces HTML-significant characters with text representations.
# The browser can then display the characters instead of treating them as tags.
safe_html_text = escape(untrusted_text, quote=True)

print("Untrusted source (not rendered):", unsafe_html_source)
print("HTML-escaped text:", safe_html_text)

# A small check: angle brackets and ampersand are represented as entities.
assert "&lt;b&gt;" in safe_html_text
assert "&amp;" in safe_html_text
print("Check passed: the sample tag is encoded for HTML text display.")

## Reflected and stored paths

- **Reflected XSS:** request-influenced text appears in a response and is rendered unsafely.
- **Stored XSS:** attacker-influenced content is saved, then rendered unsafely later, often for another user.

In an LLM application, also inspect retrieved text that the model repeats. A testimonial can be escaped on its original page yet become unsafe if a chatbot fetches and returns it through a different rendering path. Output encoding must be applied at the point of use for every destination.

## Defensive checklist

1. Identify where model output or retrieved text is rendered.
2. Insert plain text with safe text APIs whenever possible.
3. If rich HTML is a requirement, use a well-maintained HTML sanitizer with a narrow allow-list.
4. Apply encoding for the exact output context; HTML encoding does not make a value safe for SQL or shell commands.
5. Use browser policies such as Content Security Policy as an additional layer, not as a replacement for safe rendering.

A model refusal can reduce the chance that a particular payload is generated, but the application still needs secure output handling because model behavior is not an enforcement boundary.